##Step 1: Import libraries
Import libraries

In [0]:
import json
import time
import requests
from delta.tables import DeltaTable
from pyspark.sql.functions import to_timestamp, col
from databricks.sdk import WorkspaceClient

##Step 2: Create client credentials secret scope
Store client-credentials client in secret stope or set them explicitly. 

####Option 1 - Creating a secret scope using Databricks CLI:
This option requires the ability to use the Databricks CLI in the workspace terminal or in command prompt.
1. In CONNECT data services, create a client-credentials client and temporarily save the client id and secret generated. For this sample, the client-credentials client needs to be given a role that has read and write access to a CONNECT data services namespace. For instructions, see:\
https://docs.aveva.com/bundle/connect-data-services/page/1263324.html
2. Create a secret scope called `cdsscope` For instructions using the Databricks CLI, see:\
https://docs.databricks.com/en/security/secrets/index.html
3. Create two secrets within that secret scope called `cdsclientid` and `cdsclientsecret` containing the client id and secret generated from CONNECT data services. Refer to the link in step 2 for instructions.

####Option 2 - Creating a secret scope using Databricks SDK for Python:
This uses the Databricks SDK for Python option which requires entering your client credentials into the notebook in plain text. It's recommended to delete these credentials after the block is run.
1. In CONNECT data services, create a client-credentials client and temporarily save the client id and secret generated. For this sample, the client-credentials client needs to be given a role that has read and write access to a CONNECT data services namespace. For instructions, see:\
https://docs.aveva.com/bundle/connect-data-services/page/1263324.html
2. Enter your client credentials into the option 2 code block
3. Run the option 2 code block.

In [0]:
# use this to skip setting scope if already done
scopeAlreadyCreated = False

if scopeAlreadyCreated == False:
    # enter your client id and secret here
    clientId = "YOUR_CLIENT_ID"
    clientSecret = "YOUR_CLIENT_SECRET"

    w = WorkspaceClient()

    # create scope
    try:
        w.secrets.create_scope(scope="cdsscope")
        print("Created scope")
    except Exception as e:
        print(e)

    # create secrets
    try:
        w.secrets.put_secret(scope="cdsscope", key="cdsclientid", string_value=clientId)
        w.secrets.put_secret(
            scope="cdsscope", key="cdsclientsecret", string_value=clientSecret
        )
        print("Put secrets")
    except Exception as e:
        print(e)

##Step 3: Get token
Set connection information and use OAuth2.0 client credentials flow to get a bearer token.

In [0]:
# retrieve secrets from Databricks secrets
clientId = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientid")
clientSecret = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientsecret")

# set connection information
apiVersion = "v1"
resource = "https://uswe.datahub.connect.aveva.com" # change is using region other than US west
tenantId = "YOUR_TENANT_ID"
namespaceId = "YOUR_NAMESPACE_ID"

# choose streams to write to delta table
streamIds = ["Stream1"] # can specify multiple stream ids 

# use the client ID and Secret to get the needed bearer token
token_endpoint = f'{resource}/identity/connect/token'
token_information = requests.post(token_endpoint,data={'client_id': clientId,'client_secret': clientSecret,'grant_type': 'client_credentials'})
token = json.loads(token_information.content)["access_token"]
print("Bearer token is: " + token)

##Step 4: Create delta table
The next few steps will demonstrate how to sign up for, and receive updates on the data for a stream using the CONNECT data services change broker. With those updates, will update a delta table. This step in particular will demonstrate creating a new table in databricks.

In [0]:
# create a new Delta table

catalog_name = "YOUR_CATALOG_NAME"  # replace with your catalog name
schema_name = "YOUR_SCHEMA_NAME"  # replace with your schema name or "default" to use the default schema
table_name = "YOUR_TABLE_NAME" # replace with your desired table name

spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.{table_name} (
    timestamp TIMESTAMP,
    streamId STRING,
    value DOUBLE
)
USING delta
"""
)

##Step 5: Sign up for updates
Sign up for updates from the change broker for our stream of interest.

In [0]:
# define the signup input.
signupInput = {
    "Name": "Sample Databricks Get Updates from CONNECT Notebook",
    "ResourceIds": streamIds,
    "ResourceType": "Stream",
}
# make a POST request to create the signup.
response = requests.post(
    f"{resource}/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/",
    headers={"Authorization": f"Bearer {token}"},
    json=signupInput,
)
createSignup = response.json()
status = "Activating"
# loop until the signup is active
while status == "Activating":
    response = requests.get(
        f"{resource}/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{createSignup['id']}",
        headers={"Authorization": f"Bearer {token}"},
    )
    signup = response.json()
    status = signup["signupState"]
    time.sleep(1)
# record the bookmark and signupId for later use.
bookmark = signup["bookmark"]
signupId = signup["id"]
print("Signup ID is " + signupId + " and bookmark is " + bookmark)

##Step 6: Get updates and write to delta table

In [0]:
# change these to increase or decrease the number of times to get updates and the delay between each request
repeat = 3
delay = 10 #secs

def fetch_updates(bookmark):
    response = requests.get(
        f"{resource}/api/{apiVersion}/Tenants/{tenantId}/Namespaces/{namespaceId}/Signups/{signupId}/Updates?bookmark={bookmark}",
        headers={"Authorization": f"Bearer {token}"},
    )
    return response.json()

def process_update(update, dataList, deleteList, deleteWindows):
    if update["operation"] in ["Insert", "Update", "Replace"]:
        for event in update["events"]:
            print(
                str(event["Timestamp"])
                + " "
                + update["resourceId"]
                + " "
                + str(event["Value"])
                + " "
                + update["operation"]
            )
            stream = update["resourceId"]
            data = [event["Timestamp"], stream, float(event["Value"])]
            dataList.append(data)
    elif update["operation"] == "Remove":
        for event in update["events"]:
            print(
                str(event["Timestamp"])
                + " "
                + update["resourceId"]
                + " "
                + update["operation"]
            )
            stream = update["resourceId"]
            delete = [event["Timestamp"], stream]
            deleteList.append(delete)
    elif update["operation"] == "RemoveWindow":
        print(
            str(update["events"][0]["Timestamp"])
            + "-"
            + str(update["events"][1]["Timestamp"])
            + " "
            + update["resourceId"]
            + " "
            + update["operation"]
        )
        stream = update["resourceId"]
        deleteWindows.append(
            [stream, update["events"][0]["Timestamp"], update["events"][1]["Timestamp"]]
        ) 

def merge_data(dataList):
    if dataList:
        df = spark.createDataFrame(dataList, ["timestamp", "streamId", "value"])
        df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
        df.createOrReplaceTempView("updates_view")
        spark.sql(
            f"""
        MERGE INTO {catalog_name}.{schema_name}.{table_name} AS target
        USING updates_view AS source
        ON target.timestamp = source.timestamp AND target.streamId = source.streamId
        WHEN MATCHED THEN
        UPDATE SET
        target.value = source.value
        WHEN NOT MATCHED THEN
            INSERT (timestamp, streamId, value)
        VALUES (source.timestamp, source.streamId, source.value)
        """
        )

def delete_data(deleteList):
    if deleteList:
        df = spark.createDataFrame(deleteList, ["timestamp", "streamId"])
        df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
        df.createOrReplaceTempView("deletes_view")
        spark.sql(
            f"""
        DELETE FROM {catalog_name}.{schema_name}.{table_name}
        WHERE (timestamp) IN (SELECT timestamp FROM deletes_view) AND (streamId) IN (SELECT streamId FROM deletes_view)
        """
        )

def delete_windows(deleteWindows):
    if deleteWindows:
        for window in deleteWindows:
            stream = window[0]
            start = window[1]
            end = window[2]
            spark.sql(
                f"""
            DELETE FROM {catalog_name}.{schema_name}.{table_name}
            WHERE streamId = '{stream}' AND timestamp BETWEEN to_timestamp('{start}') AND to_timestamp('{end}')
            """
            )

for i in range(repeat):
    updates_data = fetch_updates(bookmark)
    updates = updates_data["data"]
    bookmark = updates_data["bookmark"]
    
    # reset lists
    dataList = []
    deleteList = []
    deleteWindows = []

    # process updates
    for update in updates:
        process_update(update, dataList, deleteList, deleteWindows)

    # merge or delete table data
    merge_data(dataList)
    delete_data(deleteList)
    delete_windows(deleteWindows)

    time.sleep(delay)

##Step 7: Check for updates in Delta Table

In [0]:
result_df = spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.{table_name} ORDER BY timestamp DESC")
display(result_df)

##Step 8: Tests
Run this block after running the above blocks to test.

In [0]:
import unittest
from unittest.mock import patch, MagicMock

test_catalog_name = catalog_name
test_schema_name = schema_name
test_table_name = "test_table"

def create_test_table():
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {test_catalog_name}.{test_schema_name}.{test_table_name} (
            timestamp TIMESTAMP,
            streamId STRING,
            value DOUBLE
        )
        USING delta
        """
    )

def drop_test_table():
    spark.sql(f"DROP TABLE IF EXISTS {test_catalog_name}.{test_schema_name}.{test_table_name}")

class TestDataHubNotebook(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        create_test_table()
        global table_name
        cls._original_table_name = table_name
        table_name = test_table_name

    @classmethod
    def tearDownClass(cls):
        drop_test_table()
        global table_name
        table_name = cls._original_table_name

    def test_process_update_insert(self):
        dataList, deleteList, deleteWindows = [], [], []
        update = {
            "operation": "Insert",
            "resourceId": "Stream1",
            "events": [{"Timestamp": "2025-12-29T12:00:00Z", "Value": 123.45}]
        }
        process_update(update, dataList, deleteList, deleteWindows)
        self.assertEqual(dataList, [["2025-12-29T12:00:00Z", "Stream1", 123.45]])
        self.assertEqual(deleteList, [])
        self.assertEqual(deleteWindows, [])

    def test_process_update_remove(self):
        dataList, deleteList, deleteWindows = [], [], []
        update = {
            "operation": "Remove",
            "resourceId": "Stream1",
            "events": [{"Timestamp": "2025-12-29T12:00:00Z"}]
        }
        process_update(update, dataList, deleteList, deleteWindows)
        self.assertEqual(deleteList, [["2025-12-29T12:00:00Z", "Stream1"]])
        self.assertEqual(dataList, [])
        self.assertEqual(deleteWindows, [])

    def test_process_update_removewindow(self):
        dataList, deleteList, deleteWindows = [], [], []
        update = {
            "operation": "RemoveWindow",
            "resourceId": "Stream1",
            "events": [
                {"Timestamp": "2025-12-29T12:00:00Z"},
                {"Timestamp": "2025-12-29T13:00:00Z"}
            ]
        }
        process_update(update, dataList, deleteList, deleteWindows)
        self.assertEqual(deleteWindows, [["Stream1", "2025-12-29T12:00:00Z", "2025-12-29T13:00:00Z"]])
        self.assertEqual(dataList, [])
        self.assertEqual(deleteList, [])

    def test_merge_data(self):
        dataList = [["2025-12-29T12:00:00Z", "Stream1", 123.45]]
        merge_data(dataList)
        df = spark.sql(f"SELECT * FROM {test_catalog_name}.{test_schema_name}.{test_table_name} WHERE streamId = 'Stream1' AND value = 123.45")
        self.assertTrue(df.count() >= 0)

    def test_delete_data(self):
        deleteList = [["2025-12-29T12:00:00Z", "Stream1"]]
        delete_data(deleteList)
        df = spark.sql(f"SELECT * FROM {test_catalog_name}.{test_schema_name}.{test_table_name} WHERE streamId = 'Stream1' AND timestamp = to_timestamp('2025-12-29T12:00:00Z')")
        self.assertTrue(df.count() == 0)

    def test_delete_windows(self):
        # Add data covering the window before testing delete_windows
        dataList = [
            ["2025-12-29T12:00:00Z", "Stream1", 111.11],
            ["2025-12-29T12:30:00Z", "Stream1", 222.22],
            ["2025-12-29T13:00:00Z", "Stream1", 333.33]
        ]
        merge_data(dataList)
        deleteWindows = [["Stream1", "2025-12-29T12:00:00Z", "2025-12-29T13:00:00Z"]]
        delete_windows(deleteWindows)
        df = spark.sql(f"SELECT * FROM {test_catalog_name}.{test_schema_name}.{test_table_name} WHERE streamId = 'Stream1' AND timestamp BETWEEN to_timestamp('2025-12-29T12:00:00Z') AND to_timestamp('2025-12-29T13:00:00Z')")
        self.assertTrue(df.count() == 0)

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)